# SR-KV consensus_update_v1 pilot

Exploratory, deliberately favourable-by-design pilot - see experiments/consensus_update_protocol.md for the pre-registered hypothesis and success criterion. Runs sanity, then the pilot only if sanity passes, then the paired-bootstrap analysis, all in one GPU session (one Kaggle slot).


In [ ]:
!nvidia-smi || echo 'no GPU'
import torch
print('cuda:', torch.cuda.is_available())


In [ ]:
import subprocess, sys

REPO = 'https://github.com/Dhruvp18/FYP-SR_KV.git'
WORKDIR = '/kaggle/working/sr-kv'
LOGFILE = '/kaggle/working/run.log'

def sh(cmd, cwd=WORKDIR, check=True):
    print('+', cmd, flush=True)
    with open(LOGFILE, 'a') as logf:
        logf.write(f'+ {cmd}\n'); logf.flush()
        proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True); logf.write(line); logf.flush()
        proc.wait()
    if check and proc.returncode != 0:
        raise SystemExit(f'FAILED ({proc.returncode}): {cmd}')
    return proc.returncode


In [ ]:
import os
if os.path.isdir(WORKDIR):
    sh('git pull -q', check=False)
else:
    sh(f'git clone -q {REPO} {WORKDIR}', cwd='/kaggle/working')
sh('git log --oneline -1')


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
sh("pip install -q -U 'transformers>=5.0' accelerate bitsandbytes", cwd='/kaggle/working')
import transformers; print('transformers', transformers.__version__)


## Restore any earlier progress

This kernel resumes from its OWN previous output via `git pull` only - unlike the
phaseN kernels it mounts no `kernel_sources`, since this pilot has no earlier phase
to build on. Re-pushing after committing partial `results/consensus_update_v1/` progress is how a killed run resumes.


## Quick CPU test suite


In [ ]:
sh('python -m pytest -q')


## Sanity set (5 conversations, all variants/methods/budgets)

Must pass before the 50-conversation pilot runs - see experiments/consensus_update_protocol.md.


In [ ]:
sh('python scripts/run_consensus_update.py --stage sanity')


In [ ]:
import json
gate = json.load(open(f'{WORKDIR}/results/consensus_update_v1/sanity_gate.json'))
print(json.dumps(gate, indent=2))
if not gate['pass']:
    raise SystemExit('Sanity gate failed - stopping before the pilot set, per protocol.')


## Pilot set (50 conversations)


In [ ]:
sh('python scripts/run_consensus_update.py --stage pilot')


## Pre-registered analysis


In [ ]:
sh('python scripts/analyse_consensus_update.py results/consensus_update_v1/pilot.jsonl')


## Results are the kernel output


In [ ]:
sh('ls -la results/consensus_update_v1 | head -40', check=False)
